In [4]:
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi

In [ ]:
import importlib
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.data_cleaning.data import _get_project_data_dir, download_data, load_data, clean_data

In [ ]:
download_data()

In [ ]:
df = load_data()
df_cleaned = clean_data(df)

In [ ]:
# pip install nltk si besoin
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
_stop_words = set(stopwords.words("english"))
_lemmatizer = WordNetLemmatizer()


def preprocess_text(text, remove_stopwords=True, remove_punctuation=True, lemmatize=True):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]

    if remove_punctuation:
        tokens = [t for t in tokens if t.isalpha()]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in _stop_words]

    if lemmatize:
        for pos in ("n", "v", "a"):
            tokens = [_lemmatizer.lemmatize(t, pos=pos) for t in tokens]

    return " ".join(tokens)

In [ ]:
_stop_words
_stop_words.remove()

In [ ]:
text = "I ate chocolate yesterday and I feel happy right now!"
preprocess_text(text)

In [ ]:
df_balanced["clean_title"] = df_balanced["title"].apply(preprocess_text)

In [ ]:
df_balanced[["title", "clean_title"]].head()

In [ ]:
df_balanced.drop(columns=["title"], inplace=True)

In [ ]:
# On sauvegarde en .csv
# df_balanced.to_csv("../../data/emotion_balanced.csv", index=False)

In [ ]:
# On sauvegarde la data en .pkl pour éviter de refaire le nettoyage à chaque fois
# df_balanced.to_pickle("../data/emotion_balanced.pkl")

In [ ]:
# TODO: compute the BOW
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

# We create the output BOW, we can even reject directly the stop words and the punctuation, how magical?
vectorizer = CountVectorizer(
    max_features=10000,
    stop_words="english",
    # min_df=0.001, max_df=0.999
)
X = vectorizer.fit_transform(df_balanced["clean_title"])  # scipy sparse matrix
print(type(X), X.shape)  # should be (num_docs, num_features)

BOW = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=df_balanced.index,
    columns=vectorizer.get_feature_names_out(),
)

In [ ]:
# On vérifie si BOW est bien une sparse matrix
sparse.issparse(BOW)

In [ ]:
df_balanced["clean_title"][899]

In [ ]:
y = df_balanced["label"]
X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# TODO: Perform a logistic regression to predict whether a message is a spam or not
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)
# 91% pour preprocess et LogisticRegression de base.

In [ ]:
######## ON REFAIT TOUT SANS BALANCE CLASSES
######## ET AVEC CLASS_WEIGHT A LA FIN

In [ ]:
df2 = load_data()
df2.shape

In [ ]:
df2_cleaned = clean_data(df2)

In [ ]:
df2_cleaned.shape

In [ ]:
sys.path.insert(0, os.path.abspath(".."))
from src.training.preprocess import preprocess_text, clean_text

In [ ]:
# On applique clean_text et preprocess_text sur title
df2_cleaned["clean_title"] = df2_cleaned["title"].apply(clean_text)
df2_cleaned["clean_title"] = df2_cleaned["clean_title"].apply(preprocess_text)

In [ ]:
df2_cleaned.drop(columns=["title"], inplace=True)

In [45]:
df2_cleaned.shape

(2470669, 2)

In [ ]:
# df2_cleaned.to_csv("../../data/emotion_preprocessed.csv", index=False)

In [ ]:
df2_cleaned = pd.read_csv("../../data/emotion_preprocessed.csv")

(2470669, 2)

In [9]:
df2_cleaned.dropna(inplace=True)

In [ ]:
"""# TODO: compute the BOW
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

# We create the output BOW, we can even reject directly the stop words and the punctuation, how magical?
vectorizer = CountVectorizer(max_features=10000,
                             #stop_words='english',
                             #min_df=0.001, max_df=0.999
)
X = vectorizer.fit_transform(df2_cleaned["clean_title"])  # scipy sparse matrix
print(type(X), X.shape)  # should be (num_docs, num_features)

BOW = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=df2_cleaned.index,
    columns=vectorizer.get_feature_names_out(),
)"""

<class 'scipy.sparse._csr.csr_matrix'> (2470669, 10000)


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# We create the output BOW, we can even reject directly the stop words and the punctuation, how magical?
vectorizer = TfidfVectorizer(
    max_features=10000,
    # stop_words='english',
    # min_df=0.001, max_df=0.999
)
X = vectorizer.fit_transform(df2_cleaned["clean_title"])  # scipy sparse matrix
print(type(X), X.shape)  # should be (num_docs, num_features)

BOW = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=df2_cleaned.index,
    columns=vectorizer.get_feature_names_out(),
)

<class 'scipy.sparse._csr.csr_matrix'> (2469119, 10000)


In [11]:
y = df2_cleaned["label"]
X.shape, y.shape

((2469119, 10000), (2469119,))

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1975295, 10000), (493824, 10000), (1975295,), (493824,))

In [40]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)

/home/thomas_d/.pyenv/versions/MHSD/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9380471693912987

In [31]:
from sklearn.metrics import classification_report

class_report = classification_report(y_test, log_reg.predict(X_test))

In [32]:
# Classification report en tableau
print(class_report)

              precision    recall  f1-score   support

         0.0       0.98      0.93      0.95    398052
         1.0       0.76      0.92      0.83     96082

    accuracy                           0.93    494134
   macro avg       0.87      0.92      0.89    494134
weighted avg       0.94      0.93      0.93    494134



In [ ]:
import joblib

# joblib.dump(log_reg, '../models/depression_log_classifier.pkl')
# joblib.dump(vectorizer, '../models/count_vectorizer.pkl')
print("✅ Modèle baseline sauvegardé !")

✅ Modèle baseline sauvegardé !


In [49]:
from src.training.predict import LogisticRegression_predict

y_pred = LogisticRegression_predict(X_train, y_train, X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.98      0.93      0.96    398052
         1.0       0.77      0.92      0.84     96082

    accuracy                           0.93    494134
   macro avg       0.87      0.93      0.90    494134
weighted avg       0.94      0.93      0.93    494134



In [2]:
import joblib

xgb_model = joblib.load("../models/xgb_depression_classifier.pkl")
vectorizer_xgb_tfidf = joblib.load("../models/xgb_tfidf_vectorizer.pkl")

In [13]:
xgb_model.score(X_test, y_test)

0.8033145412130638

In [16]:
type(y)

pandas.Series